In [ ]:
import pandas as pd

def load_dataframe_from_row6(file_path):
    
    df = pd.read_excel(
        file_path,
        header=5  # Row 6 in Excel
    )
    
    return df

In [ ]:
df = load_dataframe_from_row6("Lijst declaraties.xlsx")

df_hist = load_dataframe_from_row6("Lijst declaraties historie.xlsx")

df = df.drop(columns=["Invoer door"])




In [ ]:
df

In [ ]:
#col_idx = df.columns.get_loc("Invoerdatum")
#cols = df.columns[col_idx-8:col_idx+1]

#def shift_row_right(row):
 #   values = [v for v in row if pd.notna(v)]
  #  return [pd.NA] * (len(row) - len(values)) + values

#df[cols] = df[cols].apply(shift_row_right, axis=1, result_type='expand')

#df.replace("", pd.NA, inplace=True)

In [ ]:
df["Soort"].value_counts(dropna=False)
df = df[df["Soort"]=="ZPM"]
df["Zorggroep"].value_counts(dropna=False)


uzovi_to_zorggroep = {
    3351: "ASR",
    7037: "CZ",
    3358: "Cooperatie VGZ",
    3360: "Menzis",
    3356: "De Friesland",
    3330: "DSW",
    3369: "Multizorg VRZ"
}

In [ ]:
df_hist = df_hist[df_hist["Soort"]=="ZPM"]

In [ ]:
df["Zorggroep"] = df["Zorggroep"].fillna(df["Uzovi"].map(uzovi_to_zorggroep))
import pandas as pd

# Load file
df_plafond = pd.read_excel("ZPM-contract-plafond verzekeraars.xlsx")

df_plafond = load_dataframe_from_row6("ZPM-contract-plafond verzekeraars.xlsx")

result = (
    df_plafond.groupby("Zorggroep")["Contract-plafond"]
      .apply(lambda x: x.dropna().iloc[0] if x.notna().any() else None)
      .to_dict()
)

print(result)

In [ ]:
df['date'] = pd.to_datetime(df['Einddatum'])
df = df[df['Einddatum'].dt.year == 2026]
df.columns

In [ ]:
df['Zorggroep'].value_counts()

In [ ]:
df_hist['Zorggroep'].value_counts()

In [ ]:
df.groupby('Zorggroep')['Totaal toegekend'].mean()

In [ ]:
means = df_hist.groupby('Zorggroep')['Totaal toegekend'].mean()

valid_groups = means[means > 0].index

df_hist = df_hist[df_hist['Zorggroep'].isin(valid_groups)]

df_hist.groupby('Zorggroep')['Totaal toegekend'].mean()

In [ ]:
df

In [ ]:
df = df.copy()

df_hist = df_hist.copy()

df_hist['Einddatum'] = pd.to_datetime(df_hist['Einddatum'])

df24 = df_hist[df_hist['Einddatum'].dt.year == 2024]

df25 = df_hist[df_hist['Einddatum'].dt.year == 2025]

In [ ]:
df['Totaal toegekend'] = pd.to_numeric(df['Totaal toegekend'], errors='coerce')

df['Avg_received'] = (
    df.groupby('Zorggroep')['Totaal toegekend']
      .transform('mean')
)


df24['Avg_received'] = (
    df24.groupby('Zorggroep')['Totaal toegekend']
      .transform('mean')
)

df25['Avg_received'] = (
    df25.groupby('Zorggroep')['Totaal toegekend']
      .transform('mean')
)




In [ ]:
df['Nbr_Claims'] = (
    (df['Totaal toegekend'] > 0)
    .groupby(df['Zorggroep'])
    .transform('sum')
)

df24['Nbr_Claims'] = (
    (df24['Totaal ingediend'] > 0)
    .groupby(df24['Zorggroep'])
    .transform('sum')
)

df25['Nbr_Claims'] = (
    (df25['Totaal ingediend'] > 0)
    .groupby(df25['Zorggroep'])
    .transform('sum')
)

In [ ]:
df.groupby('Zorggroep')['Nbr_Claims'].first()

In [ ]:
df['Sum_received'] = (
    df.groupby('Zorggroep')['Totaal toegekend']
      .transform('sum')
)

df24['Sum_received'] = (
    df24.groupby('Zorggroep')['Totaal ingediend']
      .transform('sum')
)

df25['Sum_received'] = (
    df25.groupby('Zorggroep')['Totaal ingediend']
      .transform('sum')
)

In [ ]:
# ---------------- DF ----------------
df['Invoerdatum'] = pd.to_datetime(df['Invoerdatum'])
df['cumulative_claimed_tmp'] = df.groupby('Zorggroep')['Totaal toegekend'].cumsum()
df['current_trend'] = pd.NA

for group, g_df in df.groupby('Zorggroep'):
    g_df = g_df.sort_values('Invoerdatum')

    last_date = g_df['Invoerdatum'].max()
    year_start = pd.Timestamp(f'{last_date.year}-01-01')

    ytd = g_df[g_df['Invoerdatum'] >= year_start]

    if len(ytd) >= 2:
        first_date = ytd['Invoerdatum'].iloc[0]
        last_date_ytd = ytd['Invoerdatum'].iloc[-1]

        first_val = ytd['cumulative_claimed_tmp'].iloc[0]
        last_val = ytd['cumulative_claimed_tmp'].iloc[-1]

        days = (last_date_ytd - first_date).days
        trend = (last_val - first_val) / days if days != 0 else 0
    else:
        trend = 0

    df.loc[g_df.index, 'current_trend'] = trend


# ---------------- DF24 ----------------
df24['Invoerdatum'] = pd.to_datetime(df24['Invoerdatum'])
df24['cumulative_claimed_tmp'] = df24.groupby('Zorggroep')['Totaal toegekend'].cumsum()
df24['current_trend'] = pd.NA

for group, g_df in df24.groupby('Zorggroep'):
    g_df = g_df.sort_values('Invoerdatum')

    last_date = g_df['Invoerdatum'].max()
    year_start = pd.Timestamp(f'{last_date.year}-01-01')

    ytd = g_df[g_df['Invoerdatum'] >= year_start]

    if len(ytd) >= 2:
        first_date = ytd['Invoerdatum'].iloc[0]
        last_date_ytd = ytd['Invoerdatum'].iloc[-1]

        first_val = ytd['cumulative_claimed_tmp'].iloc[0]
        last_val = ytd['cumulative_claimed_tmp'].iloc[-1]

        days = (last_date_ytd - first_date).days
        trend = (last_val - first_val) / days if days != 0 else 0
    else:
        trend = 0

    df24.loc[g_df.index, 'current_trend'] = trend


# ---------------- DF25 ----------------
df25['Invoerdatum'] = pd.to_datetime(df25['Invoerdatum'])
df25['cumulative_claimed_tmp'] = df25.groupby('Zorggroep')['Totaal toegekend'].cumsum()
df25['current_trend'] = pd.NA

for group, g_df in df25.groupby('Zorggroep'):
    g_df = g_df.sort_values('Invoerdatum')

    last_date = g_df['Invoerdatum'].max()
    year_start = pd.Timestamp(f'{last_date.year}-01-01')

    ytd = g_df[g_df['Invoerdatum'] >= year_start]

    if len(ytd) >= 2:
        first_date = ytd['Invoerdatum'].iloc[0]
        last_date_ytd = ytd['Invoerdatum'].iloc[-1]

        first_val = ytd['cumulative_claimed_tmp'].iloc[0]
        last_val = ytd['cumulative_claimed_tmp'].iloc[-1]

        days = (last_date_ytd - first_date).days
        trend = (last_val - first_val) / days if days != 0 else 0
    else:
        trend = 0

    df25.loc[g_df.index, 'current_trend'] = trend


# ---------------- CLEANUP ----------------
df.drop(columns=['cumulative_claimed_tmp'], inplace=True)
df24.drop(columns=['cumulative_claimed_tmp'], inplace=True)
df25.drop(columns=['cumulative_claimed_tmp'], inplace=True)

In [ ]:
import pandas as pd

from sklearn.linear_model import (
    LinearRegression,
    Ridge,
    Lasso
)

from sklearn.ensemble import (
    RandomForestRegressor,
    GradientBoostingRegressor,
    VotingRegressor
)

from sklearn.metrics import mean_absolute_error, r2_score

# =========================
# PREP TRAIN DATA
# =========================

df_train = pd.concat([df24, df25], ignore_index=True)

X_train = df_train[['Avg_received', 'Nbr_Claims', 'Zorggroep','current_trend']]
X_train = pd.get_dummies(X_train, columns=['Zorggroep'])

y_train = df_train['Sum_received']

# =========================
# PREP TEST DATA
# =========================

X_test = df[['Avg_received', 'Nbr_Claims', 'Zorggroep', 'current_trend']]
X_test = pd.get_dummies(X_test, columns=['Zorggroep'])

# Align columns
X_test = X_test.reindex(columns=X_train.columns, fill_value=0)

# =========================
# DEFINE MODELS
# =========================

lr = LinearRegression()

ridge = Ridge(alpha=1.0)

lasso = Lasso(alpha=0.01)

rf = RandomForestRegressor(
    n_estimators=200,
    max_depth=8,
    random_state=5
)

gbr = GradientBoostingRegressor(
    n_estimators=200,
    learning_rate=0.05,
    max_depth=3,
    random_state=5
)

# =========================
# MODEL AVERAGING
# =========================

ensemble = VotingRegressor([
    ('lr', lr),
    ('ridge', ridge),
    ('lasso', lasso),
    ('rf', rf),
    ('gbr', gbr)
])

# Train ensemble
ensemble.fit(X_train, y_train)

# Predict
df = df.copy()
df['Predicted_Sum_received'] = ensemble.predict(X_test)

# =========================
# OVERALL METRICS
# =========================

mae_total = mean_absolute_error(
    df['Sum_received'],
    df['Predicted_Sum_received']
)

r2_total = r2_score(
    df['Sum_received'],
    df['Predicted_Sum_received']
)

print("TOTAL MAE:", mae_total)
print("TOTAL R2 :", r2_total)

# =========================
# PER ZORGGROEP
# =========================

for zg in df['Zorggroep'].unique():

    subset = df[df['Zorggroep'] == zg]

    mae = mean_absolute_error(
        subset['Sum_received'],
        subset['Predicted_Sum_received']
    )

    r2 = r2_score(
        subset['Sum_received'],
        subset['Predicted_Sum_received']
    )

    print(f"{zg} | MAE={mae:.2f}")

    #Scores of course wont make sense as it is start of year data, end of year helps. why keep it? later perfromance monitoring.

In [ ]:
for zg in df['Zorggroep'].unique():
    subset = df[df['Zorggroep'] == zg]
    
  
    print(zg, subset["Predicted_Sum_received"].iloc[0])

In [ ]:
# --- CONVERSIONS ---
df["Totaal toegekend"] = pd.to_numeric(df["Totaal toegekend"], errors="coerce")
df["Totaal ingediend"] = pd.to_numeric(df["Totaal ingediend"], errors="coerce")
df[["Totaal ingediend", "Totaal toegekend"]] = df[["Totaal ingediend", "Totaal toegekend"]].fillna(0)

df["Invoerdatum"] = pd.to_datetime(df["Invoerdatum"])
df['ML_Predictions'] = None
# --- SORT ---
df = df.sort_values(["Zorggroep", "Invoerdatum"])

# --- CUMULATIVE SUMS ---
df["cumulative_claimed"] = (
    df.groupby("Zorggroep")["Totaal toegekend"].cumsum()
)

df["cumulative_gevraagd"] = (
    df.groupby("Zorggroep")["Totaal ingediend"].cumsum()
)

print(df[[
    "Zorggroep",
    "Invoerdatum",
    "Totaal toegekend",
    "Totaal ingediend",
    "cumulative_claimed",
    "cumulative_gevraagd"
]])

# --- GET CEILING PER ZORGGROEP ---
ceilings = (
    df_plafond.dropna(subset=["Contract-plafond"])
      .groupby("Zorggroep")["Contract-plafond"]
      .first()
)

# --- MAP CEILING BACK ---
df["Contract-plafond"] = df["Zorggroep"].map(ceilings)

# --- CREATE NEW ROWS FOR 01-01-2026 ---
new_rows = pd.DataFrame({
    "Zorggroep": df["Zorggroep"].unique(),
    "Invoerdatum": pd.to_datetime("2026-01-01"),
    "Totaal toegekend": 0,
    "Totaal ingediend": 0,
    "cumulative_claimed": 0,
    "cumulative_gevraagd": 0,
    "ML_Predictions": 0,
    "Contract-plafond": df.drop_duplicates("Zorggroep")
    
                            .set_index("Zorggroep")["Contract-plafond"]
})

# --- APPEND NEW ROWS ---
df = pd.concat([df, new_rows], ignore_index=True)

# --- SORT AGAIN ---
df = df.sort_values(["Zorggroep", "Invoerdatum"])

In [ ]:
df

In [ ]:
df['Invoerdatum'] = pd.to_datetime(df['Invoerdatum'])

# Create the new column
df['cumulative_prediction'] = None

# List to store new rows
new_rows = []

# Process each Zorggroep
for group, g_df in df.groupby('Zorggroep', sort=False):
    g_df = g_df.sort_values('Invoerdatum')
    
    first_date = g_df['Invoerdatum'].iloc[0]
    last_date = g_df['Invoerdatum'].iloc[-1]
    first_cum = g_df['cumulative_claimed'].iloc[0]
    last_cum = g_df['cumulative_claimed'].iloc[-1]
    final_pred = g_df['Predicted_Sum_received'].iloc[-1]
    
    # Linear trend slope (per day)
    days_total = (last_date - first_date).days
    slope = (last_cum - first_cum) / days_total if days_total != 0 else 0
    
    # 1️⃣ Partial copy of last row (empty cumulative_claimed & Totaal toegekend)
    partial_row = g_df.iloc[-1].copy()
    partial_row['cumulative_claimed'] = pd.NA
    partial_row['Totaal toegekend'] = pd.NA
    partial_row['cumulative_prediction'] = last_cum
    new_rows.append(partial_row)
    
    # 2️⃣ Prediction row for 31-12-2025
    predict_date = pd.to_datetime('2026-12-31')
    days_to_predict = (predict_date - last_date).days
    predicted_cum = last_cum + slope * days_to_predict
    
    prediction_row = g_df.iloc[-1].copy()
    prediction_row['Invoerdatum'] = predict_date
    prediction_row['cumulative_claimed'] = pd.NA
    prediction_row['Totaal toegekend'] = pd.NA
    prediction_row['cumulative_gevraagd'] = pd.NA
    prediction_row['cumulative_prediction'] = predicted_cum
    prediction_row['ML_Predictions'] = final_pred	
    new_rows.append(prediction_row)

# Append the new rows
df = pd.concat([df, pd.DataFrame(new_rows)], ignore_index=True)


In [ ]:
# Sort nicely
df = df.sort_values(['Zorggroep','Invoerdatum']).reset_index(drop=True)
df = df[[
    "Zorggroep",
    "Invoerdatum",
    "Totaal toegekend",
    "cumulative_claimed",
    "Contract-plafond",
    "cumulative_gevraagd",
    "cumulative_prediction",
    "ML_Predictions"
]]

# --- EXPORT ---
df.to_excel("PREDICTIONS.xlsx", sheet_name="ZPM", index=False)

Jeugd Traject

In [ ]:
import pandas as pd

def load_dataframe_from_row6(file_path):
    
    df = pd.read_excel(
        file_path,
        header=5  # Row 6 in Excel
    )
    
    return df
df = load_dataframe_from_row6("Lijst declaraties.xlsx")

df_hist = load_dataframe_from_row6("Lijst declaraties historie.xlsx")

df = df.drop(columns=["Invoer door"])

df = df[df["Soort"]=="Jeugdtraject"]
df_hist = df_hist[df_hist["Soort"]=="Jeugdtraject"]
df['date'] = pd.to_datetime(df['Einddatum'])
df = df[df['Einddatum'].dt.year == 2026]
df.columns
df['Jeugdzorgregio-naam'].value_counts()
df_hist['Jeugdzorgregio-naam'].value_counts()
df.groupby('Jeugdzorgregio-naam')['Totaal toegekend'].mean()
means = df_hist.groupby('Jeugdzorgregio-naam')['Totaal toegekend'].mean()

valid_groups = means[means > 0].index

df_hist = df_hist[df_hist['Jeugdzorgregio-naam'].isin(valid_groups)]

df_hist.groupby('Jeugdzorgregio-naam')['Totaal toegekend'].mean()
df = df.copy()

df_hist = df_hist.copy()

df_hist['Einddatum'] = pd.to_datetime(df_hist['Einddatum'])

df24 = df_hist[df_hist['Einddatum'].dt.year == 2024]

df25 = df_hist[df_hist['Einddatum'].dt.year == 2025]
df['Totaal toegekend'] = pd.to_numeric(df['Totaal toegekend'], errors='coerce')

df['Avg_received'] = (
    df.groupby('Jeugdzorgregio-naam')['Totaal toegekend']
      .transform('mean')
)


df24['Avg_received'] = (
    df24.groupby('Jeugdzorgregio-naam')['Totaal toegekend']
      .transform('mean')
)

df25['Avg_received'] = (
    df25.groupby('Jeugdzorgregio-naam')['Totaal toegekend']
      .transform('mean')
)

df['Nbr_Claims'] = (
    (df['Totaal toegekend'] > 0)
    .groupby(df['Jeugdzorgregio-naam'])
    .transform('sum')
)

df24['Nbr_Claims'] = (
    (df24['Totaal toegekend'] > 0)
    .groupby(df24['Jeugdzorgregio-naam'])
    .transform('sum')
)

df25['Nbr_Claims'] = (
    (df25['Totaal toegekend'] > 0)
    .groupby(df25['Jeugdzorgregio-naam'])
    .transform('sum')
)
df.groupby('Jeugdzorgregio-naam')['Nbr_Claims'].first()
df['Sum_received'] = (
    df.groupby('Jeugdzorgregio-naam')['Totaal toegekend']
      .transform('sum')
)

df24['Sum_received'] = (
    df24.groupby('Jeugdzorgregio-naam')['Totaal toegekend']
      .transform('sum')
)

df25['Sum_received'] = (
    df25.groupby('Jeugdzorgregio-naam')['Totaal toegekend']
      .transform('sum')
)
# ---------------- DF ----------------
df['Invoerdatum'] = pd.to_datetime(df['Invoerdatum'])
df['cumulative_claimed_tmp'] = df.groupby('Jeugdzorgregio-naam')['Totaal toegekend'].cumsum()
df['current_trend'] = pd.NA

for group, g_df in df.groupby('Jeugdzorgregio-naam'):
    g_df = g_df.sort_values('Invoerdatum')

    last_date = g_df['Invoerdatum'].max()
    year_start = pd.Timestamp(f'{last_date.year}-01-01')

    ytd = g_df[g_df['Invoerdatum'] >= year_start]

    if len(ytd) >= 2:
        first_date = ytd['Invoerdatum'].iloc[0]
        last_date_ytd = ytd['Invoerdatum'].iloc[-1]

        first_val = ytd['cumulative_claimed_tmp'].iloc[0]
        last_val = ytd['cumulative_claimed_tmp'].iloc[-1]

        days = (last_date_ytd - first_date).days
        trend = (last_val - first_val) / days if days != 0 else 0
    else:
        trend = 0

    df.loc[g_df.index, 'current_trend'] = trend


# ---------------- DF24 ----------------
df24['Invoerdatum'] = pd.to_datetime(df24['Invoerdatum'])
df24['cumulative_claimed_tmp'] = df24.groupby('Jeugdzorgregio-naam')['Totaal toegekend'].cumsum()
df24['current_trend'] = pd.NA

for group, g_df in df24.groupby('Jeugdzorgregio-naam'):
    g_df = g_df.sort_values('Invoerdatum')

    last_date = g_df['Invoerdatum'].max()
    year_start = pd.Timestamp(f'{last_date.year}-01-01')

    ytd = g_df[g_df['Invoerdatum'] >= year_start]

    if len(ytd) >= 2:
        first_date = ytd['Invoerdatum'].iloc[0]
        last_date_ytd = ytd['Invoerdatum'].iloc[-1]

        first_val = ytd['cumulative_claimed_tmp'].iloc[0]
        last_val = ytd['cumulative_claimed_tmp'].iloc[-1]

        days = (last_date_ytd - first_date).days
        trend = (last_val - first_val) / days if days != 0 else 0
    else:
        trend = 0

    df24.loc[g_df.index, 'current_trend'] = trend


# ---------------- DF25 ----------------
df25['Invoerdatum'] = pd.to_datetime(df25['Invoerdatum'])
df25['cumulative_claimed_tmp'] = df25.groupby('Jeugdzorgregio-naam')['Totaal toegekend'].cumsum()
df25['current_trend'] = pd.NA

for group, g_df in df25.groupby('Jeugdzorgregio-naam'):
    g_df = g_df.sort_values('Invoerdatum')

    last_date = g_df['Invoerdatum'].max()
    year_start = pd.Timestamp(f'{last_date.year}-01-01')

    ytd = g_df[g_df['Invoerdatum'] >= year_start]

    if len(ytd) >= 2:
        first_date = ytd['Invoerdatum'].iloc[0]
        last_date_ytd = ytd['Invoerdatum'].iloc[-1]

        first_val = ytd['cumulative_claimed_tmp'].iloc[0]
        last_val = ytd['cumulative_claimed_tmp'].iloc[-1]

        days = (last_date_ytd - first_date).days
        trend = (last_val - first_val) / days if days != 0 else 0
    else:
        trend = 0

    df25.loc[g_df.index, 'current_trend'] = trend


# ---------------- CLEANUP ----------------
df.drop(columns=['cumulative_claimed_tmp'], inplace=True)
df24.drop(columns=['cumulative_claimed_tmp'], inplace=True)
df25.drop(columns=['cumulative_claimed_tmp'], inplace=True)
import pandas as pd

from sklearn.linear_model import (
    LinearRegression,
    Ridge,
    Lasso
)

from sklearn.ensemble import (
    RandomForestRegressor,
    GradientBoostingRegressor,
    VotingRegressor
)

from sklearn.metrics import mean_absolute_error, r2_score

# =========================
# PREP TRAIN DATA
# =========================

df_train = pd.concat([df24, df25], ignore_index=True)

X_train = df_train[['Avg_received', 'Nbr_Claims', 'Jeugdzorgregio-naam','current_trend']]
X_train = pd.get_dummies(X_train, columns=['Jeugdzorgregio-naam'])

y_train = df_train['Sum_received']

# =========================
# PREP TEST DATA
# =========================

X_test = df[['Avg_received', 'Nbr_Claims', 'Jeugdzorgregio-naam', 'current_trend']]
X_test = pd.get_dummies(X_test, columns=['Jeugdzorgregio-naam'])

# Align columns
X_test = X_test.reindex(columns=X_train.columns, fill_value=0)

# =========================
# DEFINE MODELS
# =========================

lr = LinearRegression()

ridge = Ridge(alpha=1.0)

lasso = Lasso(alpha=0.01)

rf = RandomForestRegressor(
    n_estimators=200,
    max_depth=8,
    random_state=42
)

gbr = GradientBoostingRegressor(
    n_estimators=200,
    learning_rate=0.05,
    max_depth=3,
    random_state=42
)

# =========================
# MODEL AVERAGING
# =========================

ensemble = VotingRegressor([
    ('lr', lr),
    ('ridge', ridge),
    ('lasso', lasso),
    ('rf', rf),
    ('gbr', gbr)
])

# Train ensemble
ensemble.fit(X_train, y_train)

# Predict
df = df.copy()
df['Predicted_Sum_received'] = ensemble.predict(X_test)

# =========================
# OVERALL METRICS
# =========================

mae_total = mean_absolute_error(
    df['Sum_received'],
    df['Predicted_Sum_received']
)

r2_total = r2_score(
    df['Sum_received'],
    df['Predicted_Sum_received']
)

print("TOTAL MAE:", mae_total)
print("TOTAL R2 :", r2_total)

# =========================
# PER Jeugdzorgregio-naam
# =========================

for zg in df['Jeugdzorgregio-naam'].unique():

    subset = df[df['Jeugdzorgregio-naam'] == zg]

    mae = mean_absolute_error(
        subset['Sum_received'],
        subset['Predicted_Sum_received']
    )

    r2 = r2_score(
        subset['Sum_received'],
        subset['Predicted_Sum_received']
    )

    print(f"{zg} | MAE={mae:.2f} | R2={r2:.3f}")
for zg in df['Jeugdzorgregio-naam'].unique():
    subset = df[df['Jeugdzorgregio-naam'] == zg]
    
  
    print(zg, subset["Predicted_Sum_received"].iloc[0])
# --- CONVERSIONS ---
df["Totaal toegekend"] = pd.to_numeric(df["Totaal toegekend"], errors="coerce")
df["Totaal ingediend"] = pd.to_numeric(df["Totaal ingediend"], errors="coerce")
df[["Totaal ingediend", "Totaal toegekend"]] = df[["Totaal ingediend", "Totaal toegekend"]].fillna(0)

df["Invoerdatum"] = pd.to_datetime(df["Invoerdatum"])
df['ML_Predictions'] = None
# --- SORT ---
df = df.sort_values(["Jeugdzorgregio-naam", "Invoerdatum"])

# --- CUMULATIVE SUMS ---
df["cumulative_claimed"] = (
    df.groupby("Jeugdzorgregio-naam")["Totaal toegekend"].cumsum()
)

df["cumulative_gevraagd"] = (
    df.groupby("Jeugdzorgregio-naam")["Totaal ingediend"].cumsum()
)

print(df[[
    "Jeugdzorgregio-naam",
    "Invoerdatum",
    "Totaal toegekend",
    "Totaal ingediend",
    "cumulative_claimed",
    "cumulative_gevraagd"
]])



# --- CREATE NEW ROWS FOR 01-01-2026 ---
new_rows = pd.DataFrame({
    "Jeugdzorgregio-naam": df["Jeugdzorgregio-naam"].unique(),
    "Invoerdatum": pd.to_datetime("2026-01-01"),
    "Totaal toegekend": 0,
    "Totaal ingediend": 0,
    "cumulative_claimed": 0,
    "cumulative_gevraagd": 0,
    "ML_Predictions": 0
})

# --- APPEND NEW ROWS ---
df = pd.concat([df, new_rows], ignore_index=True)

# --- SORT AGAIN ---
df = df.sort_values(["Jeugdzorgregio-naam", "Invoerdatum"])
df['Invoerdatum'] = pd.to_datetime(df['Invoerdatum'])

# Create the new column
df['cumulative_prediction'] = None

# List to store new rows
new_rows = []
df['Empty'] = None

# Process each Jeugdzorgregio-naam
for group, g_df in df.groupby('Jeugdzorgregio-naam', sort=False):
    g_df = g_df.sort_values('Invoerdatum')
    
    first_date = g_df['Invoerdatum'].iloc[0]
    last_date = g_df['Invoerdatum'].iloc[-1]
    first_cum = g_df['cumulative_claimed'].iloc[0]
    last_cum = g_df['cumulative_claimed'].iloc[-1]
    final_pred = g_df['Predicted_Sum_received'].iloc[-1]
    
    # Linear trend slope (per day)
    days_total = (last_date - first_date).days
    slope = (last_cum - first_cum) / days_total if days_total != 0 else 0
    
    # 1️⃣ Partial copy of last row (empty cumulative_claimed & Totaal toegekend)
    partial_row = g_df.iloc[-1].copy()
    partial_row['cumulative_claimed'] = pd.NA
    partial_row['Totaal toegekend'] = pd.NA
    partial_row['cumulative_prediction'] = last_cum
    new_rows.append(partial_row)
    
    # 2️⃣ Prediction row for 31-12-2025
    predict_date = pd.to_datetime('2026-12-31')
    days_to_predict = (predict_date - last_date).days
    predicted_cum = last_cum + slope * days_to_predict
    
    prediction_row = g_df.iloc[-1].copy()
    prediction_row['Invoerdatum'] = predict_date
    prediction_row['cumulative_claimed'] = pd.NA
    prediction_row['Totaal toegekend'] = pd.NA
    prediction_row['cumulative_gevraagd'] = pd.NA
    prediction_row['cumulative_prediction'] = predicted_cum
    prediction_row['ML_Predictions'] = abs(final_pred)	
    new_rows.append(prediction_row)

# Append the new rows
df = pd.concat([df, pd.DataFrame(new_rows)], ignore_index=True)

# Sort nicely
df = df.sort_values(['Jeugdzorgregio-naam','Invoerdatum']).reset_index(drop=True)
df = df[[
    "Jeugdzorgregio-naam",
    "Invoerdatum",
    "Totaal toegekend",
    "cumulative_claimed",
    "cumulative_gevraagd",
    "cumulative_prediction",
    "ML_Predictions",
    "Empty"
]]

# --- EXPORT ---
df.to_excel("Jeugd.xlsx", sheet_name="Jeugd", index=False)

Rejection analysis


In [ ]:
import pandas as pd
import os

excel_file = "Lijst prestaties JW323 met retourinformatie.xlsx"
final_csv = "Final_Rejected_Analysis.csv"

def run_full_preparation():
    try:
        if not os.path.exists(excel_file):
            print(f"❌ Error: Cannot find '{excel_file}'")
            return

        print(f"Reading {excel_file}...")

        # Step 1: Detect header row
        found_df = None
        for skip in range(10):
            temp_df = pd.read_excel(excel_file, skiprows=skip)
            temp_df.columns = [str(c).strip() for c in temp_df.columns]

            if any("Fac.-nr" in c for c in temp_df.columns):
                found_df = temp_df
                break

        if found_df is None:
            print("❌ Error: Could not find columns.")
            return

        df = found_df

        # Step 2: Identify columns
        status_col = next((c for c in df.columns if "status" in c.lower()), None)
        retour_cols = [c for c in df.columns if "retourcode" in c.lower()]
        fixed_cols = ["Fac.-nr", "Jeugdzorgregio", "Gemeente"]
        available_fixed = [c for c in fixed_cols if c in df.columns]

        # Step 3: Filter for '3-Geweigerd'
        df_base = df[df[status_col].astype(str).str.contains('3-Geweigerd', na=False)].copy()

        # Step 4: Create the "Searchable" version (Melted)
        df_final = df_base.melt(
            id_vars=available_fixed + [status_col],
            value_vars=retour_cols,
            var_name='Bron_Kolom',
            value_name='Raw_Value'
        )

        # Remove empty rows
        df_final = df_final.dropna(subset=['Raw_Value'])
        df_final = df_final[df_final['Raw_Value'].astype(str).str.strip() != '']

        # Step 5: Split Code and Omschrijving
        print("Cleaning text and extracting codes...")
        split_data = df_final['Raw_Value'].astype(str).str.split('-', n=1, expand=True)

        df_final['Retourcode_Nr'] = split_data[0].str.strip()
        df_final['Retour_Omschrijving'] = split_data[1].str.strip().fillna(df_final['Raw_Value'])

        # Remove the messy raw column
        df_final = df_final.drop(columns=['Raw_Value'])

        # Step 6: Save
        df_final.to_csv(final_csv, index=False, sep=';', encoding='utf-8-sig')

        print("-" * 30)
        print(f"✅ SUCCESS! Created: {final_csv}")
        print(f"Ready for Searchable Slicer in Power BI.")
        print("-" * 30)

    except Exception as e:
        print(f"❌ Error: {e}")

if __name__ == "__main__":
    run_full_preparation()